# TP2a — Régression linéaire avec PyTorch

Nous refaisons le TP1 — le prix d'un appartement en fonction de sa surface — avec **PyTorch**,
la bibliothèque que nous utiliserons pour les réseaux de neurones.

Nous remplaçons votre code du TP1 par PyTorch, une brique à la fois, en vérifiant à chaque fois que l'on obtient le même résultat.

# Les données

Les mêmes ventes qu'au TP1. Exécutez la cellule.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn

df = pd.read_csv('data/bordeaux_appartements_2025.csv.gz')

cols = {'valeur_fonciere': 'prix', 'surface_reelle_bati': 'surface'}
examples = df.rename(columns=cols, errors='raise')[['prix', 'surface']]
examples

# 1. Les tenseurs : plus besoin de boucle

Un **tenseur** (*tensor*) est un tableau de nombres, comme une liste Python. Sa **forme** (*shape*) donne
le nombre d'éléments dans chaque dimension. Le nombre de crochets imbriqués donne le nombre de dimensions.

In [ ]:
t1 = torch.tensor([45.0, 60.0, 100.0])          # 1 dimension : 3 nombres
t2 = torch.tensor([[45.0], [60.0], [100.0]])    # 2 dimensions : 3 lignes, 1 colonne
t3 = torch.tensor([[45.0]])                     # 2 dimensions : 1 ligne, 1 colonne

print('t1.shape :', t1.shape)
print('t2.shape :', t2.shape)
print('t3.shape :', t3.shape)
print(t2)

Un modèle PyTorch attend toujours des données à **2 dimensions** : **une ligne par exemple, une colonne par caractéristique**.
Pour 3 appartements dont on ne connaît que la surface, c'est la forme de `t2` : `(3, 1)`.

Créez le tenseur `two_flats` pour 2 appartements, de 45 m² et 70 m². Il doit avoir la forme `(2, 1)`.

In [ ]:
# TODO un tenseur de forme (2, 1)
two_flats = ...
print('two_flats.shape :', two_flats.shape)

Nos données sont rangées de la même façon : une ligne par appartement, une colonne pour la surface.

In [ ]:
X = torch.tensor(examples[['surface']].values, dtype=torch.float32)
Y = torch.tensor(examples[['prix']].values, dtype=torch.float32)

print('X.shape :', X.shape)
print('Y.shape :', Y.shape)
print(X[:3])         # les 3 premières lignes

Une opération sur un tenseur s'applique à **tous** ses éléments d'un coup, sans boucle.

In [ ]:
print(X[:3] * 2)            # chaque surface est multipliée par 2
print(X.mean())             # la moyenne des 3 109 surfaces : un seul nombre
print(X[:3] - X.mean())     # on retranche la moyenne à chaque surface

Au TP1a, la méthode `cost()` parcourait les appartements avec une boucle `for`.
Reprenons le modèle trouvé à la main ($w$ = 4 000, $b$ = 10 000), et calculez la même chose **sans boucle** :

- `Y_pred` : la prédiction $wx + b$ pour tous les appartements ;
- `cost` : la moyenne des carrés des écarts entre `Y_pred` et `Y`. Un tenseur a une méthode `.mean()`.

In [ ]:
w, b = 4000, 10000

# TODO la prédiction pour tous les appartements
Y_pred = ...

# TODO le coût (MSE)
cost = ...

print(f'RMSE : {cost.item() ** 0.5:,.0f} €')

Vous devez retrouver la RMSE du TP1 pour ce modèle (environ 77 000 €).

# 2. `nn.Linear` et `nn.MSELoss` : votre modèle et votre coût, tout prêts

PyTorch fournit déjà ces deux calculs :

| TP1 | PyTorch |
|-----|---------|
| `LinearModel.f` : $wx + b$ | `model = nn.Linear(1, 1)` puis `model(X)` |
| `LinearModel.cost` : la MSE | `cost_function = nn.MSELoss()` puis `cost_function(Y_pred, Y)` |

Vérifions-le en donnant à `nn.Linear` les mêmes $w$ et $b$.

In [ ]:
# 1 entrée (la surface), 1 sortie (le prix). La couche contient un poids w et un biais b.
model = nn.Linear(in_features=1, out_features=1)

# On impose nos valeurs à la place des valeurs tirées au hasard par PyTorch
nn.init.constant_(model.weight, w)
nn.init.constant_(model.bias, b)

cost_function = nn.MSELoss()
Y_pred_torch = model(X)
cost_torch = cost_function(Y_pred_torch, Y)

print('prédictions, à la main :', Y_pred[:3].flatten().tolist())
print('prédictions, PyTorch   :', Y_pred_torch[:3].flatten().tolist())
print(f'RMSE à la main : {cost.item() ** 0.5:,.0f} €')
print(f'RMSE PyTorch   : {cost_torch.item() ** 0.5:,.0f} €')

# 3. Le modèle complet

Comme à la fin du TP1b, le modèle doit travailler sur des surfaces **standardisées**.
Plutôt que de standardiser à la main avant chaque utilisation, on range la standardisation **dans le modèle**.

Un modèle PyTorch est une classe qui hérite de `nn.Module`, avec deux méthodes :

- `__init__` : ce que contient le modèle. Ici, la moyenne et l'écart-type des surfaces d'entraînement, et une couche `nn.Linear` ;
- `forward` : le calcul de la prédiction, appelé quand on écrit `model(X)`.

Lisez la classe, et complétez la ligne `self.linear` : la couche `nn.Linear` vue dans la partie 2.

In [ ]:
class LinearModel(nn.Module):
    def __init__(self, X):
        super().__init__()
        # Moyenne et écart-type calculés une fois pour toutes sur les données d'entraînement
        self.mean = X.mean(dim=0)
        self.std = X.std(dim=0)
        # TODO la couche linéaire : 1 entrée (la surface), 1 sortie (le prix)
        self.linear = ...

    def standardize(self, x):
        # moyenne 0, écart-type 1 : on retranche la moyenne, puis on divise par l'écart-type
        return (x - self.mean) / self.std

    def forward(self, x):
        # on standardise x, puis on applique la couche linéaire
        return self.linear(self.standardize(x))


torch.manual_seed(0)
model = LinearModel(X)

In [ ]:
# Rappel : la standardisation ramène la surface à une moyenne de 0 et un écart-type de 1
X_scaled = model.standardize(X)
print(f'surface brute        : moyenne {X.mean():6.1f}   écart-type {X.std():5.1f}   de {X.min():5.1f} à {X.max():5.1f}')
print(f'surface standardisée : moyenne {X_scaled.mean():6.1f}   écart-type {X_scaled.std():5.1f}   de {X_scaled.min():5.1f} à {X_scaled.max():5.1f}')
print()
for surface, scaled in zip(X[:3].flatten(), X_scaled[:3].flatten()):
    print(f'{surface:5.0f} m²  →  {scaled:+.2f}')

# 4. L'entraînement

Il reste à remplacer la descente de gradient du TP1b. Chaque epoch enchaîne toujours les **4 mêmes étapes**,
quel que soit le modèle :

| Étape | PyTorch | TP1b |
|-------|---------|------|
| 1. prédiction | `Y_pred = model(X)` | `model.f(x)` |
| 2. coût | `cost = cost_function(Y_pred, Y)` | `model.cost(X, Y)` |
| 3. calcul des pentes | `cost.backward()` | `dJ_dw`, `dJ_db` |
| 4. mise à jour : un pas dans la descente | `optimizer.step()` puis `optimizer.zero_grad()` | `update_model` |

Complétez la fonction `train()`.

In [ ]:
def train(model, X, Y, epochs, alpha):
    cost_function = nn.MSELoss()
    # SGD : la descente de gradient. PyTorch appelle lr (learning rate) le taux d'apprentissage alpha du TP1
    optimizer = torch.optim.SGD(model.parameters(), lr=alpha)

    for epoch in range(epochs):
        # TODO les 4 étapes
        Y_pred = ...
        cost = ...




        if epoch % 10 == 0:
            print(f'epoch: {epoch}, RMSE: {cost.item() ** 0.5:,.0f} €')
    print(f'RMSE finale : {cost.item() ** 0.5:,.0f} €')

In [ ]:
# On repart d'un modèle neuf, aux paramètres tirés au hasard, à chaque exécution de la cellule
torch.manual_seed(0)
model = LinearModel(X)

# Le modèle standardise lui-même : on lui donne les surfaces brutes
train(model, X, Y, epochs=60, alpha=0.1)

## Questions

1. Au TP1b, la descente de gradient sur données standardisées atteignait une RMSE d'environ **77 000 €**.
   Retrouvez-vous la même valeur ? Au bout de combien d'epochs ?
2. Essayez `alpha = 0.001`, puis `alpha = 1.5`. Que se passe-t-il ? Remettez ensuite `alpha = 0.1` et réexécutez la cellule.

## Réponses

TODO


# 5. L'inférence

Le modèle est entraîné : on peut maintenant l'utiliser pour prédire le prix d'un appartement qu'il n'a jamais vu.
C'est l'**inférence** (*inference*).

Pendant l'entraînement, PyTorch garde la trace de chaque calcul pour pouvoir calculer les pentes avec `backward()`.
En inférence, c'est inutile : on place les appels au modèle dans un bloc `with torch.no_grad():`.

Que prédit le modèle pour un appartement de **45 m²** ? Et pour **46 m²** ? Déduisez-en le prix d'un m² supplémentaire.
Le modèle attend un tenseur de forme `(1, 1)` : une ligne (un appartement), une colonne (la surface).

In [ ]:
with torch.no_grad():
    # TODO appliquer le modèle à 45 m², puis à 46 m²
    price_45 = ...
    price_46 = ...

print(f'45 m² : {price_45.item():,.0f} €')
print(f'46 m² : {price_46.item():,.0f} €')
print(f'prix du m² supplémentaire : {(price_46 - price_45).item():,.0f} €')

Le tracé de la droite est lui aussi une inférence : on demande au modèle le prix de 10 surfaces entre 0 et 250 m².

In [ ]:
surfaces = torch.linspace(0, 250, 10).reshape(-1, 1)   # 10 surfaces, forme (10, 1)
with torch.no_grad():
    prices = model(surfaces)

ax = examples.plot.scatter(x='surface', y='prix', s=1)
ax.plot(surfaces, prices, color='black')
plt.show()

## Question

3. Au TP1b, le modèle trouvé avait un `weight` d'environ **3 800 €**. Comparez avec le prix du m² supplémentaire
   calculé ci-dessus. Pourquoi est-ce le même nombre ?

## Réponse

TODO


# La suite

Au TP2b, on remplace la couche linéaire par un réseau de neurones. La fonction `train()` ne change pas :

```python
model = nn.Sequential(
    nn.Linear(4, 30),   # couche cachée : 30 neurones
    nn.ReLU(),          # fonction d'activation
    nn.Linear(30, 1),   # couche de sortie : le prix
)
```